In [2]:
!pip install z3-solver==4.12.2.0

In [3]:
# ============================================================
# FILTER EXPERIMENT 02 — CLEAN STANDALONE
# Matched compute: CONTROL vs FILTER vs PLACEBO (3 seeds)
# MIT-compatible. Zero proprietary references.
# ============================================================
import os, json, time, random, hashlib, platform
from copy import deepcopy
from enum import Enum
import numpy as np
import torch
import torch.nn.functional as F
from pydantic import BaseModel, ValidationError
from transformers import AutoTokenizer, AutoModelForCausalLM

EXPERIMENT = "Filter Experiment 02"
SEEDS = [11, 22, 33]
EPOCHS = 4
UPDATES_PER_EPOCH = 84
LR = 5e-5
MAX_LENGTH = 64
GRAD_CLIP = 1.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "/kaggle/working/filter_exp02"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_HASHES = {
    "model.safetensors": "c7d00560d8910fbed77ffad4065dee5011c41ba401b1064e749c498ba9e20373",
    "config.json": "337e7106d8f04da30d6ef1617faa51369cddaa34a855bf76844c9798b46b26e8",
    "vocab.json": "f6bd25a65e4e63ca31360e9fb11c7e4f9a391a78385d640acd814092dd6eee4f",
    "merges.txt": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
}

GROUND_TRUTH = {
    ("entity_a", "color"): "blue", ("entity_b", "color"): "green",
    ("entity_c", "color"): "yellow", ("entity_d", "color"): "purple",
    ("entity_e", "shape"): "circle", ("entity_f", "shape"): "square",
    ("entity_g", "shape"): "triangle", ("entity_h", "shape"): "hexagon",
    ("entity_i", "code"): "17", ("entity_j", "code"): "29",
    ("entity_k", "code"): "41", ("entity_l", "code"): "53",
}

SYNTHETIC_DATA = [
    {"subject": "entity_c", "predicate": "color", "value": "yellow"},
    {"subject": "entity_d", "predicate": "color", "value": "purple"},
    {"subject": "entity_g", "predicate": "shape", "value": "triangle"},
    {"subject": "entity_h", "predicate": "shape", "value": "hexagon"},
    {"subject": "entity_k", "predicate": "code", "value": "41"},
    {"subject": "entity_l", "predicate": "code", "value": "53"},
    {"subject": "entity_a", "predicate": "color", "value": "red"},
    {"subject": "entity_b", "predicate": "color", "value": "orange"},
    {"subject": "entity_e", "predicate": "shape", "value": "triangle"},
    {"subject": "entity_f", "predicate": "shape", "value": "circle"},
    {"subject": "entity_i", "predicate": "code", "value": "99"},
    {"subject": "entity_j", "predicate": "code", "value": "88"},
    {"subject": "entity_x", "predicate": "color", "value": "silver"},
    {"subject": "entity_y", "predicate": "shape", "value": "star"},
]

CONTAMINATION_TEST = [
    ("entity_a", "color", "blue", "red"), ("entity_b", "color", "green", "orange"),
    ("entity_e", "shape", "circle", "triangle"), ("entity_f", "shape", "square", "circle"),
    ("entity_i", "code", "17", "99"), ("entity_j", "code", "29", "88"),
]

class Claim(BaseModel):
    subject: str; predicate: str; value: str
class Verdict(str, Enum):
    VERIFIED="VERIFIED"; CONTRADICTED="CONTRADICTED"; UNKNOWN="UNKNOWN"; INVALID="INVALID"

class FilterGate:
    def __init__(self, gt):
        self.gt = {(s.strip().lower(), p.strip().lower()): str(v).strip().lower() for (s,p),v in gt.items()}
    def audit(self, raw):
        try: c = Claim(**raw)
        except: return Verdict.INVALID
        key = (c.subject.strip().lower(), c.predicate.strip().lower())
        val = c.value.strip().lower()
        if key not in self.gt: return Verdict.UNKNOWN
        return Verdict.VERIFIED if val == self.gt[key] else Verdict.CONTRADICTED

def fact_text(r): return f"FACT: {r['subject']} {r['predicate']} = {r['value']}"
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
def sha256_file(p):
    h = hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda: f.read(1024*1024), b""): h.update(c)
    return h.hexdigest()
def cycle_to_length(items, length, seed):
    out = []
    while len(out) < length: out.extend(items)
    out = out[:length]; random.Random(seed).shuffle(out); return out

@torch.no_grad()
def continuation_score(model, prefix, continuation):
    model.eval()
    p_ids = tokenizer(prefix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    f_ids = tokenizer(prefix+continuation, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    logits = model(f_ids).logits[:, :-1, :]
    labels = f_ids[:, 1:]
    lp = F.log_softmax(logits, dim=-1)
    sel = torch.gather(lp, -1, labels.unsqueeze(-1)).squeeze(-1)
    return float(sel[:, p_ids.shape[1]-1:].mean().item())

def evaluate(model, label):
    rows = []
    for subj, pred, truth, false in CONTAMINATION_TEST:
        prefix = f"FACT: {subj} {pred} ="
        ts = continuation_score(model, prefix, " "+truth)
        fs = continuation_score(model, prefix, " "+false)
        rows.append({"branch": label, "subject": subj, "truth_margin": ts-fs, "prefers_truth": (ts-fs)>0})
    return rows

def train_branch(model, texts, label):
    model.train(); opt = torch.optim.AdamW(model.parameters(), lr=LR)
    for _ in range(EPOCHS):
        for text in texts:
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
            opt.zero_grad(); out = model(**enc, labels=enc["input_ids"]); loss = out.loss
            if not torch.isfinite(loss): raise RuntimeError(f"{label}: non-finite")
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); opt.step()

# --- MAIN ---
model_candidates, tokenizer_candidates = [], []
for root, _, files in os.walk("/kaggle/input"):
    f = set(files)
    if "config.json" in f and "model.safetensors" in f: model_candidates.append(root)
    if "vocab.json" in f and "merges.txt" in f: tokenizer_candidates.append(root)
MODEL_PATH, TOKENIZER_PATH = model_candidates[0], tokenizer_candidates[0]
for fname, exp in EXPECTED_HASHES.items():
    p = os.path.join(MODEL_PATH if "model" in fname or "config" in fname else TOKENIZER_PATH, fname)
    assert sha256_file(p) == exp, f"Hash mismatch: {fname}"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, local_files_only=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True).to(DEVICE)
gate = FilterGate(GROUND_TRUTH)

control_base = [fact_text(r) for r in SYNTHETIC_DATA]
filter_base = [fact_text(r) for r in SYNTHETIC_DATA if gate.audit(r) == Verdict.VERIFIED]

all_runs = []
for seed in SEEDS:
    set_seed(seed)
    placebo_base = [fact_text(r) for r in random.Random(seed+777).sample(SYNTHETIC_DATA, 6)]
    streams = {
        "CONTROL": cycle_to_length(control_base, UPDATES_PER_EPOCH, seed),
        "FILTER": cycle_to_length(filter_base, UPDATES_PER_EPOCH, seed),
        "PLACEBO": cycle_to_length(placebo_base, UPDATES_PER_EPOCH, seed),
    }
    models = {k: deepcopy(base).to(DEVICE) for k in streams}
    for k, txts in streams.items(): train_branch(models[k], txts, k)
    res = {k: evaluate(m, k) for k, m in models.items()}
    margins = {k: np.mean([r["truth_margin"] for r in res[k]]) for k in res}
    all_runs.append({"seed": seed, "margins": margins, "results": res})

agg = {k: {"mean_margin": float(np.mean([r["margins"][k] for r in all_runs])),
           "std_margin": float(np.std([r["margins"][k] for r in all_runs], ddof=1))} for k in ["CONTROL","FILTER","PLACEBO"]}
result = {"experiment": EXPERIMENT, "status": "completed", "seeds": SEEDS, "aggregate": agg, "runs": all_runs, "independent": True, "license_suggested": "MIT"}
out_file = os.path.join(OUTPUT_DIR, "filter_exp02_results.json")
with open(out_file, "w") as f: json.dump(result, f, indent=2, ensure_ascii=False)
print("EXP02 COMPLETE | SHA256:", sha256_file(out_file))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


EXP02 COMPLETE | SHA256: 4b3d424f308943ce41c3d6c8f11b8c99130e1eb39fb380ab4f42e7ed1705947e
